# Tâche 5 — Pipeline de collecte reproductible

**Projet 1 · African Language Data & LLM Readiness**

**Consigne** : construire le pipeline d'extraction reproductible — journal, nouvelles tentatives, versions, collecte reprenable ; données brutes séparées du texte nettoyé. KPI : 100 % des textes avec provenance, validés automatiquement.

Le pipeline est dans `src/` et se lance par une seule commande, depuis la racine du projet :

| Commande | Effet |
|---|---|
| `python src/collecter.py --langues wo yo nqo ha am` | run sur quelques éditions |
| `python src/collecter.py --tout` | les 68 éditions retenues + l'échantillon français |
| `python src/collecter.py --reprendre` | reprend le dernier run interrompu là où il s'était arrêté |

Ce notebook lance ces commandes et affiche les résultats. Réglages : section `pipeline` de `config/config.yaml`.

**Où vont les fichiers**

| Dossier | Contenu |
|---|---|
| `data/raw/dumps/<date>/` | dumps Wikipédia (brut) |
| `data/raw/api/parse/<code>/` | réponses brutes de l'API pour les pages lues dans la page affichée |
| `data/raw/api/fr/` | échantillon français : liste tirée et réponses brutes de l'API |
| `data/raw/api/siteinfo/` | informations de chaque wiki |
| `data/processed/collecte/<run>/` | `corpus/`, `provenance/`, `exclusions/`, `mesures/` (un fichier par langue), `run.log`, `journal_requetes.jsonl`, `etat.json`, `run_metadata.json`, copie de la configuration |

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\akota\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
COLLECTE = ROOT / "data" / "processed" / "collecte"


def lancer(*args):
    # Lance src/collecter.py et affiche son journal au fur et à mesure.
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.Popen([sys.executable, "-u", str(ROOT / "src" / "collecter.py"), *args], cwd=ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        p.terminate()
        print("\nRun arrêté. Pour continuer là où il s'est arrêté : lancer('--reprendre')")
    return p.wait()


def dernier_run():
    return sorted(COLLECTE.glob("*T*Z"))[-1]


def resultats(run=None):
    run = Path(run) if run else dernier_run()
    etat = json.loads((run / "etat.json").read_text(encoding="utf-8"))
    mesures = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((run / "mesures").glob("*.json"))]
    t = pd.DataFrame([{k: m[k] for k in ("langue", "pages_ns0", "textes", "exclusions", "pages_lues_api", "textes_api",
                                          "mots_total", "mots_mediane", "duree_s")}
                      | {"valides": m["validation"]["lignes_valides"], "avec_provenance": m["validation"]["avec_provenance"]}
                      for m in mesures]).set_index("langue") if mesures else pd.DataFrame()
    print("Run", run.name, ":", etat["statut"])
    for nom, ok in (etat.get("controles") or {}).items():
        print(f"  {nom} : {'OK' if ok else 'NON'}")
    for c, v in etat["langues"].items():
        if v["statut"] != "terminée":
            print(f"  {c} : {v['statut']} {v.get('erreur', '')}")
    return t

## Étape 1 — Run de test sur les 5 langues du prototype

Les dumps du 1er octobre 2026 sont déjà dans `data/raw/dumps/20261001/` : ils ne sont pas retéléchargés.

In [5]:
lancer("--langues", "wo", "yo", "nqo", "ha", "am")
resultats()

2026-11-02T10:56:58 INFO Nouveau run 20261102T105658Z : 5 édition(s)
2026-11-02T10:57:10 INFO 5 dump(s) à traiter, 229.3 Mo au total ; ordre : nqo wo am yo ha
2026-11-02T10:57:22 INFO nqo terminée : 1726 textes, 8 écartées, 812329 mots, validation 1726/1726 (11 s)
2026-11-02T10:57:34 INFO wo terminée : 1809 textes, 3 écartées, 608966 mots, validation 1809/1809 (11 s)
2026-11-02T11:18:38 INFO am : 5000 pages traitées, 4986 textes
2026-11-02T11:19:18 INFO am : 10000 pages traitées, 9931 textes
2026-11-02T11:19:31 INFO am : 15000 pages traitées, 14928 textes
2026-11-02T11:19:45 INFO am terminée : 16286 textes, 85 écartées, 1913680 mots, validation 16286/16286 (1328 s)
2026-11-02T11:26:03 INFO yo : 5000 pages traitées, 4659 textes
2026-11-02T11:26:35 INFO yo : 10000 pages traitées, 9649 textes
2026-11-02T11:26:38 INFO yo : 15000 pages traitées, 14648 textes
2026-11-02T11:26:38 INFO yo : 20000 pages traitées, 19648 textes
2026-11-02T11:26:39 INFO yo : 25000 pages traitées, 24648 textes
2026

,pages_ns0,textes,exclusions,pages_lues_api,textes_api,mots_total,mots_mediane,duree_s,valides,avec_provenance
langue,,,,,,,,,,
am,16371,16286,85,2137,2102,1913680,22.0,1328,16286,16286
ha,123826,123651,175,51,39,64034554,329.0,449,123651,123651
nqo,1734,1726,8,3,3,812329,169.0,11,1726,1726
wo,1812,1809,3,7,6,608966,50.0,11,1809,1809
yo,42398,41803,595,691,541,5637920,11.0,566,41803,41803


## Étape 2 — Comparaison avec le prototype (run 20261027T154013Z)

Le pipeline applique les mêmes règles que le prototype : on doit retrouver les mêmes textes. Comparaison par empreinte SHA-256 de chaque texte.

In [6]:
import hashlib

def empreintes(fichier):
    with open(fichier, encoding="utf-8") as f:
        return [hashlib.sha256(json.loads(l)["text"].encode("utf-8")).hexdigest() for l in f]

PROTO = ROOT / "data" / "processed" / "prototype" / "20261027T154013Z" / "corpus"
test = dernier_run()
lignes = []
for c in ["wo", "yo", "nqo", "ha", "am"]:
    a, b = empreintes(PROTO / f"{c}.jsonl"), empreintes(test / "corpus" / f"{c}.jsonl")
    sa, sb = set(a), set(b)
    lignes.append({"langue": c, "textes prototype": len(a), "textes pipeline": len(b), "communs": len(sa & sb),
                   "seulement prototype": len(sa - sb), "seulement pipeline": len(sb - sa), "fichiers identiques": a == b})
pd.DataFrame(lignes).set_index("langue")

,textes prototype,textes pipeline,communs,seulement prototype,seulement pipeline,fichiers identiques
langue,,,,,,
wo,1809,1809,1809,0,0,True
yo,41803,41803,41802,1,1,False
nqo,1726,1726,1726,0,0,True
ha,123651,123651,123651,0,0,True
am,16286,16286,16283,3,3,False


## Étape 3 — Run complet : 68 éditions + échantillon français

Ordre : des dumps les plus petits aux plus grands, puis l'échantillon français (100 000 articles tirés au hasard). Le run peut durer plusieurs heures (arabe égyptien, arabe standard).
S'il s'arrête (coupure, PC éteint, cellule interrompue), passer à l'étape 4 : rien n'est perdu au-delà du dernier point de reprise (toutes les 5 000 pages).

In [ ]:
lancer("--tout")
resultats()

2026-11-05T06:50:25 INFO Nouveau run 20261105T065025Z : 68 édition(s) + échantillon français
2026-11-05T06:50:32 WARNING réseau : ConnectionError sur https://aa.wikipedia.org/w/api.php, nouvelle tentative dans 1 s
2026-11-05T06:50:33 WARNING réseau : ConnectionError sur https://aa.wikipedia.org/w/api.php, nouvelle tentative dans 2 s
2026-11-05T06:50:35 WARNING réseau : ConnectionError sur https://aa.wikipedia.org/w/api.php, nouvelle tentative dans 4 s


## Étape 4 — Reprendre un run interrompu

Reprend le dernier run non terminé, avec la même configuration. Les langues en échec (par exemple après une coupure trop longue) sont retentées.

In [2]:
lancer("--reprendre")
resultats()

2026-11-05T20:34:24 INFO Reprise du run 20261105T065025Z
2026-11-05T20:34:24 INFO 68 dump(s) à traiter, 3204.9 Mo au total ; ordre : kr hz kj aa ng ak nup sg kaj din ve ann nr bm rn ti tig kg ee fon kai kbp bol ki ss knc guw nqo wo pcm gur kus ts om mos ny igl fat ln st xh nso kcg kab shi sn dga zgh tw zu am lg rw tn gpe ary tum so ff yo mg dag sw ig af ha arz ar
2026-11-05T20:34:24 INFO ary : reprise après la page n° 9999 (9954 textes déjà gardés)
2026-11-05T20:34:54 INFO ary : 15000 pages traitées, 14954 textes
2026-11-05T20:34:54 INFO ary : 20000 pages traitées, 19954 textes
2026-11-05T20:34:59 INFO ary terminée : 20321 textes, 46 écartées, 3954132 mots, validation 20321/20321 (91 s)
2026-11-05T20:35:11 WARNING téléchargement coupé (ConnectionError) : tumwiki-20261001-pages-articles.xml.bz2, reprise dans 30 s
2026-11-05T20:35:58 INFO tum : tumwiki-20261001-pages-articles.xml.bz2 téléchargé (21.6 Mo)
2026-11-05T20:36:13 INFO tum : 5000 pages traitées, 4997 textes
2026-11-05T20:36:32 

,pages_ns0,textes,exclusions,pages_lues_api,textes_api,mots_total,mots_mediane,duree_s,valides,avec_provenance
langue,,,,,,,,,,
aa,1,0,1,1,0,0,0.0,5,0,0
af,131205,131092,113,5,3,39811263,116.0,256,131092,131092
ak,1,1,0,0,0,17,17.0,4,1,1
am,16371,16286,85,2137,2079,1913680,22.0,57,16286,16286
ann,495,494,1,2,1,89187,106.5,6,494,494
...,...,...,...,...,...,...,...,...,...,...
wo,1812,1809,3,7,6,608966,50.0,7,1809,1809
xh,3111,3090,21,20,5,538536,84.0,4330,3090,3090
yo,42398,41803,595,691,275,5637920,11.0,106,41803,41803


## Étape 5 — Résultats d'un run

Mesures par langue, contrôles (validation du schéma, 100 % des textes avec provenance, chaque page comptée) et langues en échec.

In [4]:
resultats()

Run 20261105T065025Z : terminé
  corpus valide et 100 % des textes avec provenance : OK
  chaque page comptée (C2) : OK
  chaque langue demandée a des textes (C1) : NON


,pages_ns0,textes,exclusions,pages_lues_api,textes_api,mots_total,mots_mediane,duree_s,valides,avec_provenance
langue,,,,,,,,,,
aa,1,0,1,1,0,0,0.0,5,0,0
af,131205,131092,113,5,3,39811263,116.0,256,131092,131092
ak,1,1,0,0,0,17,17.0,4,1,1
am,16371,16286,85,2137,2079,1913680,22.0,57,16286,16286
ann,495,494,1,2,1,89187,106.5,6,494,494
...,...,...,...,...,...,...,...,...,...,...
wo,1812,1809,3,7,6,608966,50.0,7,1809,1809
xh,3111,3090,21,20,5,538536,84.0,4330,3090,3090
yo,42398,41803,595,691,275,5637920,11.0,106,41803,41803
